# Laptop Price Prediction
## Step 7: Model Evaluation
**Student:** Shivakumar | B.Tech CSE | VEMU

In [ ]:
!pip install xgboost scikit-learn joblib -q
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
import joblib, os, warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error
sns.set_theme(style='whitegrid')
plt.rcParams.update({'figure.dpi':120,'axes.titlesize':12})
print("Libraries imported")

---
## 1. Load Data & Train All Models

In [ ]:
df = pd.read_csv('laptop_price_featured.csv')
print(f'Dataset: {df.shape[0]} rows x {df.shape[1]} cols')

In [ ]:
TARGET    = 'log_price'
DROP_COLS = ['Price_euros','log_price','Ram','Weight','Cpu','Gpu','Memory','ScreenResolution','OpSys']
drop_existing = [c for c in DROP_COLS if c in df.columns]
X = df.drop(columns=drop_existing)
y = df[TARGET]
y_actual = df['Price_euros']

numerical_cols   = X.select_dtypes(include=['int64','float64']).columns.tolist()
categorical_cols = X.select_dtypes(include='object').columns.tolist()
binary_cols      = [c for c in numerical_cols if X[c].nunique() == 2]
numerical_cols   = [c for c in numerical_cols if c not in binary_cols]

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)
_,_,y_train_actual,y_test_actual = train_test_split(X,y_actual,test_size=0.2,random_state=42)

preprocessor = ColumnTransformer([
    ('num', Pipeline([('imp',SimpleImputer(strategy='median')),('sc',StandardScaler())]), numerical_cols),
    ('cat', Pipeline([('imp',SimpleImputer(strategy='most_frequent')),('enc',OneHotEncoder(handle_unknown='ignore',sparse_output=False))]), categorical_cols),
    ('bin','passthrough',binary_cols),
], remainder='drop')

X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc  = preprocessor.transform(X_test)

selector = SelectKBest(f_regression, k=20)
X_train_sel = selector.fit_transform(X_train_proc, y_train)
X_test_sel  = selector.transform(X_test_proc)

ridge = Ridge(alpha=10.0, random_state=42)
ridge.fit(X_train_sel, y_train)

rf = RandomForestRegressor(n_estimators=200, min_samples_split=5, min_samples_leaf=2, random_state=42, n_jobs=-1)
rf.fit(X_train_sel, y_train)

xgb = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6,
                    subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1,
                    reg_lambda=1.0, random_state=42, verbosity=0)
xgb.fit(X_train_sel, y_train, eval_set=[(X_test_sel,y_test)], verbose=False)

y_pred_ridge = np.expm1(ridge.predict(X_test_sel))
y_pred_rf    = np.expm1(rf.predict(X_test_sel))
y_pred_xgb   = np.expm1(xgb.predict(X_test_sel))

models_names = ['Ridge','Random Forest','XGBoost']
colors = ['#2E5D9E','#27AE60','#C55A11']
preds  = [y_pred_ridge, y_pred_rf, y_pred_xgb]
print("All 3 models trained")

---
## 2. Regression Metrics

In [ ]:
def get_metrics(y_true, y_pred, name):
    mae  = mean_absolute_error(y_true, y_pred)
    mse  = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2   = r2_score(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred)*100
    return {'Model':name,'MAE':mae,'MSE':mse,'RMSE':rmse,'R2':r2,'MAPE(%)':mape}

results = [get_metrics(y_test_actual,y_pred_ridge,'Ridge Regression'),
           get_metrics(y_test_actual,y_pred_rf,   'Random Forest'),
           get_metrics(y_test_actual,y_pred_xgb,  'XGBoost')]
metrics_df = pd.DataFrame(results).set_index('Model')

print("="*65)
print("   REGRESSION METRICS - TEST SET (Price in EUR)")
print("="*65)
display(metrics_df.round(3))
print("\nMAE  = Avg EUR error | RMSE = penalizes big errors | R2 = variance explained | MAPE = % error")

---
## 3. Metrics Bar Charts

In [ ]:
fig, axes = plt.subplots(2,2,figsize=(14,10))

metric_data = {
    'MAE (EUR)'  : [metrics_df.loc[m,'MAE']  for m in ['Ridge Regression','Random Forest','XGBoost']],
    'RMSE (EUR)' : [metrics_df.loc[m,'RMSE'] for m in ['Ridge Regression','Random Forest','XGBoost']],
    'R2 Score'   : [metrics_df.loc[m,'R2']   for m in ['Ridge Regression','Random Forest','XGBoost']],
    'MAPE (%)'   : [metrics_df.loc[m,'MAPE(%)'] for m in ['Ridge Regression','Random Forest','XGBoost']],
}

for ax, (title, vals) in zip(axes.flat, metric_data.items()):
    bars = ax.bar(models_names, vals, color=colors, edgecolor='white', alpha=0.9)
    ax.set_title(title, fontweight='bold')
    ax.bar_label(bars, fmt='%.3f', padding=3, fontsize=9)
    if 'R2' in title:
        ax.set_ylim(0,1.1)
        ax.axhline(0.85, color='red', linestyle='--', linewidth=1.5, label='Target 0.85')
        ax.legend()

plt.suptitle('Model Evaluation Metrics - Test Set', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('eval_01_metrics.png', bbox_inches='tight')
plt.show()

---
## 4. Actual vs Predicted

In [ ]:
fig, axes = plt.subplots(1,3,figsize=(18,5))
r2s = [metrics_df.loc[m,'R2'] for m in ['Ridge Regression','Random Forest','XGBoost']]

for ax, name, pred, color, r2 in zip(axes, models_names, preds, colors, r2s):
    ax.scatter(y_test_actual, pred, alpha=0.4, s=20, color=color, edgecolors='none')
    mn = min(y_test_actual.min(), pred.min())
    mx = max(y_test_actual.max(), pred.max())
    ax.plot([mn,mx],[mn,mx],'r--',linewidth=2,label='Perfect fit')
    ax.set_title(f'{name}\nActual vs Predicted', fontweight='bold')
    ax.set_xlabel('Actual Price (EUR)')
    ax.set_ylabel('Predicted Price (EUR)')
    ax.text(0.05,0.92,f'R2 = {r2:.4f}',transform=ax.transAxes,
        fontsize=11,fontweight='bold',color=color,
        bbox=dict(boxstyle='round,pad=0.3',facecolor='white',alpha=0.8))
    ax.legend(fontsize=8)

plt.suptitle('Actual vs Predicted (EUR) - Test Set', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('eval_02_actual_vs_pred.png', bbox_inches='tight')
plt.show()

---
## 5. Residual Analysis

In [ ]:
fig, axes = plt.subplots(2,3,figsize=(18,10))

for i,(name,pred,color) in enumerate(zip(models_names,preds,colors)):
    residuals = y_test_actual.values - pred
    axes[0,i].scatter(pred, residuals, alpha=0.4, s=18, color=color, edgecolors='none')
    axes[0,i].axhline(0,color='red',linestyle='--',linewidth=1.5)
    axes[0,i].set_title(f'{name} - Residuals vs Predicted', fontweight='bold')
    axes[0,i].set_xlabel('Predicted (EUR)')
    axes[0,i].set_ylabel('Residual (EUR)')

    axes[1,i].hist(residuals, bins=35, color=color, edgecolor='white', alpha=0.85)
    axes[1,i].axvline(0,color='red',linestyle='--',linewidth=1.5)
    axes[1,i].axvline(np.mean(residuals),color='orange',linewidth=1.5,label=f'Mean: {np.mean(residuals):.1f}')
    axes[1,i].set_title(f'{name} - Residual Distribution', fontweight='bold')
    axes[1,i].set_xlabel('Residual (EUR)')
    axes[1,i].legend(fontsize=8)

plt.suptitle('Residual Analysis - All Models', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('eval_03_residuals.png', bbox_inches='tight')
plt.show()

---
## 6. Cross-Validation Detail

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = {name: cross_val_score(m, X_train_sel, y_train, cv=cv, scoring='r2')
             for name, m in zip(models_names, [ridge, rf, xgb])}

fig, axes = plt.subplots(1,2,figsize=(14,5))
fold_labels = [f'Fold {i+1}' for i in range(5)]
for name, color in zip(models_names, colors):
    axes[0].plot(fold_labels, cv_scores[name], 'o-', label=name, color=color, linewidth=2, markersize=7)
axes[0].set_title('R2 Score per CV Fold', fontweight='bold')
axes[0].set_ylabel('R2 Score')
axes[0].set_ylim(0,1)
axes[0].legend()

bp = axes[1].boxplot([cv_scores[n] for n in models_names], labels=models_names,
    patch_artist=True, medianprops=dict(color='black',linewidth=2))
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color); patch.set_alpha(0.7)
axes[1].set_title('CV R2 Distribution', fontweight='bold')
axes[1].set_ylabel('R2 Score')

plt.suptitle('5-Fold Cross Validation', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('eval_04_cv.png', bbox_inches='tight')
plt.show()

print("CV Results:")
for name in models_names:
    s = cv_scores[name]
    print(f"  {name:<20}: mean={s.mean():.4f}  std={s.std():.4f}")

---
## 7. XGBoost Sample Predictions

In [ ]:
sample_idx = np.random.choice(len(y_test_actual), 15, replace=False)
sample_actual = y_test_actual.values[sample_idx]
sample_pred   = y_pred_xgb[sample_idx]
error_abs = np.abs(sample_actual - sample_pred)
error_pct = error_abs / sample_actual * 100

sample_df = pd.DataFrame({
    'Actual (EUR)'   : sample_actual.round(2),
    'Predicted (EUR)': sample_pred.round(2),
    'Error (EUR)'    : error_abs.round(2),
    'Error %'        : error_pct.round(2)
}).sort_values('Actual (EUR)')
display(sample_df)

fig, ax = plt.subplots(figsize=(12,5))
x = np.arange(len(sample_df))
ax.bar(x-0.2, sample_df['Actual (EUR)'], 0.4, label='Actual', color='#2E5D9E', alpha=0.85)
ax.bar(x+0.2, sample_df['Predicted (EUR)'], 0.4, label='Predicted', color='#C55A11', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels([f'S{i+1}' for i in range(len(sample_df))], fontsize=8)
ax.set_title('XGBoost - Actual vs Predicted (15 Samples)', fontweight='bold')
ax.set_ylabel('Price (EUR)')
ax.legend()
plt.tight_layout()
plt.savefig('eval_05_samples.png', bbox_inches='tight')
plt.show()

---
## 8. Error by Price Range

In [ ]:
error_df = pd.DataFrame({
    'Actual': y_test_actual.values,
    'Pred'  : y_pred_xgb,
    'Error' : np.abs(y_test_actual.values - y_pred_xgb)
})
bins   = [0,500,1000,1500,2000,10000]
labels = ['<500','500-1k','1k-1.5k','1.5k-2k','>2k']
error_df['Range'] = pd.cut(error_df['Actual'], bins=bins, labels=labels)

fig, axes = plt.subplots(1,2,figsize=(14,5))
range_mae = error_df.groupby('Range', observed=True)['Error'].mean()
axes[0].bar(range_mae.index, range_mae.values, color='#C55A11', edgecolor='white', alpha=0.85)
axes[0].set_title('XGBoost MAE by Price Range', fontweight='bold')
axes[0].set_xlabel('Price Range (EUR)')
axes[0].set_ylabel('MAE (EUR)')
for i,v in enumerate(range_mae.values):
    axes[0].text(i, v+5, f'{v:.0f}', ha='center', fontsize=9, fontweight='bold')

range_count = error_df.groupby('Range', observed=True).size()
axes[1].bar(range_count.index, range_count.values, color='#2E5D9E', edgecolor='white', alpha=0.85)
axes[1].set_title('Test Samples by Price Range', fontweight='bold')
axes[1].set_xlabel('Price Range (EUR)')
axes[1].set_ylabel('Count')
for i,v in enumerate(range_count.values):
    axes[1].text(i, v+1, str(v), ha='center', fontsize=9, fontweight='bold')

plt.suptitle('Error Analysis by Price Segment', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('eval_06_error_range.png', bbox_inches='tight')
plt.show()

---
## 9. Final Evaluation Summary

In [ ]:
best_mod = metrics_df['R2'].idxmax()
print("="*65)
print("         FINAL MODEL EVALUATION SUMMARY")
print("="*65)
print(f"  {'Metric':<10} {'Ridge':>15} {'Random Forest':>15} {'XGBoost':>15}")
print("  "+"-"*57)
for metric in ['MAE','MSE','RMSE','R2','MAPE(%)']:
    vals = [metrics_df.loc[m,metric] for m in ['Ridge Regression','Random Forest','XGBoost']]
    print(f"  {metric:<10} {vals[0]:>15.3f} {vals[1]:>15.3f} {vals[2]:>15.3f}")
print("="*65)
print(f"  Best Model : {best_mod}")
print(f"  Best R2    : {metrics_df['R2'].max():.4f}")
print(f"  Best MAE   : EUR {metrics_df['MAE'].min():.2f}")
print(f"  Best MAPE  : {metrics_df.loc[best_mod,'MAPE(%)']:.2f}%")
print("\n  Next Step -> Model Serialization (Step 8)")

metrics_df.round(4).to_csv('evaluation_results.csv')
from google.colab import files
files.download('evaluation_results.csv')
print("evaluation_results.csv downloaded")